# Legal Retrieval Pipeline - Minimal Runbook

In [ ]:
!scp -r /kaggle/input/datasets/dlvinh/baai-stage/* /kaggle/working/

In [ ]:
from pathlib import Path
import os
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print(Path.cwd())

## 1. Install dependencies

In [ ]:
!python3 -m pip install -r requirements.txt -q
!python3 -m pip install sentence-transformers faiss-cpu torch -q
!python3 -m pip install "unsloth[colab-new]" bitsandbytes -q

In [ ]:
DATASET_NAME = "legalraw_sample"

CORPUS_PATH = "raw_data/legalraw/full/legal_corpus.json"
QUESTIONS_PATH = "raw_data/legalraw/full/train.json"

# CORPUS_PATH = "raw_data/legalraw/sample/legal_corpus.json"
# QUESTIONS_PATH = "raw_data/legalraw/sample/train.json"


OUTPUT_DIR = "outputs"

# DENSE_MODEL =  "AITeamVN/Vietnamese_Embedding" 
# RERANK_MODEL = "AITeamVN/Vietnamese_Reranker"

DENSE_MODEL =   "BAAI/bge-m3"
RERANK_MODEL =  "BAAI/bge-reranker-v2-m3" 

LLM_RERANK_MODEL = "unsloth/Qwen3.5-9B"
LLM_RERANK_BACKEND = "unsloth_causal_lm"

USE_LLM_RERANK = True
DEVICE = "cuda"
BATCH_SIZE = 32
TOP_K = 200
CANDIDATE_TOP_K = 20
LLM_RERANK_TOP_K = 10
POSITIVE_CHUNKS_PER_AID = 2


BGE_GRAD_ACCUM = 8
RERANK_GRAD_ACCUM = 8
max_chunk_tokens = 690

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
import sys

# Add workspace root to path
workspace_root = Path.cwd()
sys.path.insert(0, str(workspace_root))

from src.utils.artifact import prepared_dir, retrieval_dir, read_table, read_json
from src.data.loaders import load_questions, load_articles
from src.data.chunking import token_count
from config import Config
from transformers import AutoTokenizer

# Load BGE tokenizer
print("Loading BGE tokenizer...")
bge_tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

def count_tokens_bge(text):
    """Count tokens using BGE tokenizer"""
    if not text:
        return 0
    tokens = bge_tokenizer.encode(text, add_special_tokens=False)
    return len(tokens)

# Create config with actual data file paths
config = Config(
    stage="evaluate",
    dataset_name="legalraw_sample",
    corpus_path=workspace_root / "raw_data" / "legalraw" / "full" / "legal_corpus.json",
    questions_path=workspace_root / "raw_data" / "legalraw" / "full" / "train.json",
    output_dir=workspace_root / "outputs",
    force=False,
    seed=42,
    
    dense_model="BAAI/bge-m3",
    rerank_model="BAAI/bge-reranker-v2-m3",
    qwen_model="Qwen/Qwen3-Reranker-0.6B",
    use_qwen_rerank=False,
    llm_rerank_model="unsloth/Qwen3.5-4B-Base",
    llm_rerank_backend="unsloth_causal_lm",
    use_llm_rerank=False,
    llm_rerank_top_k=20,
    llm_rerank_train_batch_size=1,
    llm_rerank_batch_size=16,
    llm_rerank_grad_accum=2,
    llm_rerank_epochs=1,
    llm_rerank_lr=2e-5,
    llm_rerank_max_length=630,
    llm_rerank_max_train_examples=12000,
    llm_rerank_lora_r=16,
    llm_rerank_lora_alpha=16,
    llm_rerank_load_in_4bit=True,
    device="cuda",
    batch_size=32,
    bge_train_batch_size=16,
    bge_epochs=2,
    bge_lr=2e-5,
    bge_warmup_ratio=0.1,
    bge_max_length=522,
    bge_max_train_examples=0,
    bge_use_amp=True,
    bge_gradient_checkpointing=True,
    bge_auto_batch_reduce=True,
    bge_negatives_per_example=3,
    bge_grad_accum=1,
    reranker_train_batch_size=4,
    reranker_epochs=1,
    reranker_lr=2e-5,
    reranker_warmup_ratio=0.1,
    reranker_max_length=582,
    reranker_max_train_examples=0,
    reranker_use_amp=True,
    reranker_grad_accum=2,
    
    bm25_k1=1.2,
    bm25_b=0.9,
    bm25_k1_grid="1.2",
    bm25_b_grid="0.9",
    bm25_tune_metric="recall@20",
    use_tuned_bm25=True,
    hybrid_alpha=0.5,
    alpha_grid="0.2,0.3,0.4,0.5,0.6,0.7,0.8",
    router_model="ridge",
    top_k=200,
    candidate_top_k=50,
    positive_chunks_per_aid=2,
    threshold=0.75,
    
    train_ratio=0.70,
    router_train_ratio=0.10,
    val_ratio=0.10,
    test_ratio=0.10,
    max_chunk_tokens=512,
    chunk_overlap_sentences=1,
    max_chunks=0,
    max_chunks_per_article=0,
    
    corpus_law_id_field="law_id",
    corpus_articles_field="content",
    article_id_field="aid",
    article_text_field="content_Article",
    question_id_field="qid",
    question_text_field="question",
    relevant_ids_field="relevant_laws",
)

print(f"Working directory: {workspace_root}")
print(f"Corpus path: {config.corpus_path}")
print(f"Questions path: {config.questions_path}")
print(f"BGE tokenizer loaded successfully")

In [ ]:
# Load articles
articles = load_articles(config)
print(f"Total articles: {len(articles)}")

# Calculate token counts for each article using BGE tokenizer
article_tokens = []
for i, article in enumerate(articles):
    if i % 1000 == 0:
        print(f"Processing article {i}/{len(articles)}...")
    tokens = count_tokens_bge(article['text'])
    article_tokens.append({
        'aid': article['aid'],
        'law_id': article.get('law_id', ''),
        'token_count': tokens,
        'text_length': len(article['text'])
    })

articles_df = pd.DataFrame(article_tokens)

# Statistics
print("\n=== ARTICLE CORPUS STATISTICS (BGE Tokenizer) ===")
print(f"Number of articles: {len(articles_df)}")
print(f"Total tokens: {articles_df['token_count'].sum():,}")
print(f"Max tokens: {articles_df['token_count'].max():,}")
print(f"Mean tokens: {articles_df['token_count'].mean():.2f}")
print(f"Median (p50): {articles_df['token_count'].median():.2f}")
print(f"P90: {articles_df['token_count'].quantile(0.90):.2f}")
print(f"P95: {articles_df['token_count'].quantile(0.95):.2f}")
print(f"P99: {articles_df['token_count'].quantile(0.99):.2f}")

articles_df.head(10)

In [ ]:
# Load chunks
chunks_rows = read_table(prepared_dir(config) / 'chunks.parquet')
chunks_df = pd.DataFrame(chunks_rows)
print(f"Total chunks: {len(chunks_df)}")

# Calculate token counts using BGE tokenizer
print("Calculating token counts for chunks...")
chunks_df['token_count'] = chunks_df['text'].apply(count_tokens_bge)

# Statistics
print("\n=== CHUNK CORPUS STATISTICS (BGE Tokenizer) ===")
print(f"Number of chunks: {len(chunks_df)}")
print(f"Total tokens: {chunks_df['token_count'].sum():,}")
print(f"Max tokens: {chunks_df['token_count'].max():,}")
print(f"Mean tokens: {chunks_df['token_count'].mean():.2f}")
print(f"Median (p50): {chunks_df['token_count'].median():.2f}")
print(f"P90: {chunks_df['token_count'].quantile(0.90):.2f}")
print(f"P95: {chunks_df['token_count'].quantile(0.95):.2f}")
print(f"P99: {chunks_df['token_count'].quantile(0.99):.2f}")

print(f"\nAverage chunks per article: {len(chunks_df) / len(articles_df):.2f}")

In [ ]:
# Load questions
questions = load_questions(config)
print(f"Total questions: {len(questions)}")

# Calculate token counts and relevant_laws count using BGE tokenizer
question_stats = []
for i, q in enumerate(questions):
    if i % 500 == 0:
        print(f"Processing question {i}/{len(questions)}...")
    tokens = count_tokens_bge(q['question'])
    relevant_count = len(q.get('relevant_laws', []))
    question_stats.append({
        'qid': q['qid'],
        'token_count': tokens,
        'relevant_laws_count': relevant_count,
        'text_length': len(q['question'])
    })

questions_df = pd.DataFrame(question_stats)

# Statistics
print("\n=== QUESTION CORPUS STATISTICS (BGE Tokenizer) ===")
print(f"Number of questions: {len(questions_df)}")
print(f"Total tokens: {questions_df['token_count'].sum():,}")
print(f"Max tokens: {questions_df['token_count'].max():,}")
print(f"Mean tokens: {questions_df['token_count'].mean():.2f}")
print(f"Median (p50): {questions_df['token_count'].median():.2f}")
print(f"P90: {questions_df['token_count'].quantile(0.90):.2f}")
print(f"P95: {questions_df['token_count'].quantile(0.95):.2f}")
print(f"P99: {questions_df['token_count'].quantile(0.99):.2f}")

print(f"\nRelevant laws per question:")
print(f"  Mean: {questions_df['relevant_laws_count'].mean():.2f}")
print(f"  Median: {questions_df['relevant_laws_count'].median():.2f}")
print(f"  Max: {questions_df['relevant_laws_count'].max()}")

questions_df.head(10)

In [ ]:
# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Token Count Distribution across Corpora', fontsize=16, fontweight='bold')

# Article histogram
axes[0, 0].hist(articles_df['token_count'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Token Count')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title(f'Article Tokens (n={len(articles_df)}, mean={articles_df["token_count"].mean():.0f})')
axes[0, 0].grid(axis='y', alpha=0.3)

# Chunk histogram
axes[0, 1].hist(chunks_df['token_count'], bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Token Count')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title(f'Chunk Tokens (n={len(chunks_df)}, mean={chunks_df["token_count"].mean():.0f})')
axes[0, 1].grid(axis='y', alpha=0.3)

# Question histogram
axes[1, 0].hist(questions_df['token_count'], bins=30, color='seagreen', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Token Count')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title(f'Question Tokens (n={len(questions_df)}, mean={questions_df["token_count"].mean():.0f})')
axes[1, 0].grid(axis='y', alpha=0.3)

# Relevant laws distribution
axes[1, 1].hist(questions_df['relevant_laws_count'], bins=range(0, int(questions_df['relevant_laws_count'].max())+2), 
                 color='mediumpurple', edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Number of Relevant Laws')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title(f'Relevant Laws per Question (mean={questions_df["relevant_laws_count"].mean():.2f})')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
# LƯU Ý: Phải savefig TRƯỚC khi show
plt.savefig('corpus_token_distribution.png', dpi=300, bbox_inches='tight')

# Hiển thị ảnh ra màn hình (nếu chạy trên Jupyter Notebook/VS Code)
plt.show()

print("Histogram saved as 'corpus_token_distribution.png'")

In [ ]:
# Load splits
splits_path = prepared_dir(config) / 'splits.json'
if splits_path.exists():
    splits = read_json(splits_path)
    print("\n=== DATASET SPLITS ===")
    for split_name, qids in splits.items():
        print(f"{split_name}: {len(qids)} questions")
else:
    print("Splits file not found")

In [ ]:
# Check for negative samples and training data
retrieval_dir_path = retrieval_dir(config)

training_stats = {
    'merged_scores.parquet': 'Merged BM25 + BGE scores',
    'merged_chunks.parquet': 'Merged chunks (chunk-level)',
}

print("\n=== TRAINING/RETRIEVAL DATA ===")
for filename, description in training_stats.items():
    filepath = retrieval_dir_path / filename
    if filepath.exists():
        data = read_table(filepath)
        unique_qids = data['qid'].nunique() if 'qid' in data.columns else 0
        unique_aids = data['aid'].nunique() if 'aid' in data.columns else 0
        print(f"\n{filename}:")
        print(f"  Total rows: {len(data):,}")
        print(f"  Unique questions: {unique_qids}")
        print(f"  Unique articles/items: {unique_aids}")
        print(f"  Avg items per question: {len(data) / unique_qids:.2f}")
        print(f"  Description: {description}")
    else:
        print(f"\n{filename}: Not found")

In [ ]:
# Create summary table
summary_data = {
    'Corpus': ['Articles', 'Chunks', 'Questions'],
    'Count': [
        len(articles_df),
        len(chunks_df),
        len(questions_df)
    ],
    'Total Tokens': [
        f"{articles_df['token_count'].sum():,}",
        f"{chunks_df['token_count'].sum():,}",
        f"{questions_df['token_count'].sum():,}"
    ],
    'Max Tokens': [
        articles_df['token_count'].max(),
        chunks_df['token_count'].max(),
        questions_df['token_count'].max()
    ],
    'Mean Tokens': [
        f"{articles_df['token_count'].mean():.2f}",
        f"{chunks_df['token_count'].mean():.2f}",
        f"{questions_df['token_count'].mean():.2f}"
    ],
    'P50 (Median)': [
        f"{articles_df['token_count'].median():.2f}",
        f"{chunks_df['token_count'].median():.2f}",
        f"{questions_df['token_count'].median():.2f}"
    ],
    'P90': [
        f"{articles_df['token_count'].quantile(0.90):.2f}",
        f"{chunks_df['token_count'].quantile(0.90):.2f}",
        f"{questions_df['token_count'].quantile(0.90):.2f}"
    ],
    'P95': [
        f"{articles_df['token_count'].quantile(0.95):.2f}",
        f"{chunks_df['token_count'].quantile(0.95):.2f}",
        f"{questions_df['token_count'].quantile(0.95):.2f}"
    ],
    'P99': [
        f"{articles_df['token_count'].quantile(0.99):.2f}",
        f"{chunks_df['token_count'].quantile(0.99):.2f}",
        f"{questions_df['token_count'].quantile(0.99):.2f}"
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n=== COMPREHENSIVE SUMMARY ===")
print(summary_df.to_string(index=False))

# Save to CSV
summary_df.to_csv('corpus_summary_statistics.csv', index=False)
print("\nSummary saved to 'corpus_summary_statistics.csv'")

In [ ]:
print("\n=== ADDITIONAL INSIGHTS ===")
print(f"\nChunk efficiency:")
print(f"  Articles compressed to chunks: {len(articles_df)} -> {len(chunks_df)} ({len(chunks_df)/len(articles_df):.1f}x increase)")
print(f"  Chunk size control: max_chunk_tokens = {config.max_chunk_tokens}")
print(f"  Chunk overlap: {config.chunk_overlap_sentences} sentences")

print(f"\nToken efficiency:")
total_article_tokens = articles_df['token_count'].sum()
total_chunk_tokens = chunks_df['token_count'].sum()
overlap_overhead = ((total_chunk_tokens - total_article_tokens) / total_article_tokens * 100) if total_article_tokens > 0 else 0
print(f"  Total article tokens: {total_article_tokens:,}")
print(f"  Total chunk tokens: {total_chunk_tokens:,}")
print(f"  Overlap overhead: {overlap_overhead:.2f}%")

print(f"\nQuestion characteristics:")
print(f"  Average relevant laws: {questions_df['relevant_laws_count'].mean():.2f}")
print(f"  Max relevant laws: {questions_df['relevant_laws_count'].max()}")
print(f"  Questions with 1 relevant law: {(questions_df['relevant_laws_count'] == 1).sum()}")
print(f"  Questions with 5+ relevant laws: {(questions_df['relevant_laws_count'] >= 5).sum()}")

In [ ]:
import json

# Mở và đọc file JSON
with open('/kaggle/input/datasets/dlvin755/legalraw/legal_corpus.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

In [ ]:
len(data)